# W12 Hands-on: How an LLM works, with real Gemma
# W12 實作：用真的 Gemma 看懂 LLM 怎麼運作

**How to use this notebook / 怎麼用這本筆記本**

- Each grey box is a code cell. Click it, then press **Shift + Enter** (or the ▶ button on the left) to run it. Run cells from top to bottom.
  每個灰色框是一格程式。點一下，按 **Shift + Enter**（或左邊的 ▶）執行。請由上往下依序跑。
- Or use the menu **Runtime → Run all** to run everything at once (about 2–4 minutes on a free CPU).
  也可以用選單 **執行階段 → 全部執行** 一次跑完（免費 CPU 大約 2–4 分鐘）。
- You do **not** need any account key or token. Everything downloads from public pages.
  **不需要**任何帳號金鑰或 token，所有檔案都從公開網頁下載。
- If you see "Warning: You are sending unauthenticated requests", that is normal. Ignore it.
  看到 "Warning: You are sending unauthenticated requests" 是正常的，不用理它。
- The first time, Colab asks you to sign in to Google and warns that this notebook was not written by Google. Click **Run anyway** to continue.
  第一次開會要登入 Google，並出現「這本筆記本不是 Google 寫的」提醒，按繼續執行即可。
- Keep this tab open for the whole class. You will come back to it from later steps on the website; do not reopen it.
  這個分頁整堂課留著，後面幾站切回來看，不要重開。

**Today's five steps / 今天的五步地圖**: the main sentence is *"Hualien is famous for"*; Steps 2 and 3 also use extra examples (cat, trophy).
主線例句是 *"Hualien is famous for"*；第 2、3 段另外用 cat 與 trophy 的例子。

| Step | Question | 這一步回答什麼 |
|---|---|---|
| 1 Tokenization | How does text become numbers? | 文字怎麼變成數字？ |
| 2 Embeddings | How do numbers carry meaning? | 數字怎麼帶有意思？ |
| 3 Attention | How does a word look at other words? | 一個字怎麼「看」其他字？ |
| 4 Sampling | How is the next word chosen? | 下一個字怎麼選出來？（取樣） |
| 5 Chat | How does a chat become one long text? | 聊天怎麼變成一長串文字？ |

**Which tokenizer and model? / 用的是哪個斷詞器、哪個模型？**

- Step 1 and the first half of Step 5 use the **real Gemma 4 tokenizer** (`google/gemma-4-E2B-it`, the same one the official guide uses). We only load the Gemma 4 tokenizer (we do not download the 10 GB model).
  第 1 步和第 5 步前半用**真的 Gemma 4 斷詞器**（跟官方導覽同一個）。只載入 Gemma 4 的斷詞器（不下載 10 GB 的模型）。
- Steps 2–5 run a **smaller model of the same family**: Gemma 3 270M (`unsloth/gemma-3-270m-it`, a public copy of `google/gemma-3-270m-it`). The ideas are the same, but **the numbers are different** from the official Gemma 4 guide.
  第 2–5 步跑的是**比較小的同家族模型** Gemma 3 270M。道理一樣，但**數字跟官方 Gemma 4 導覽不一樣**。

## Setup: download and load / 準備：下載與載入

Run this once. It takes about 1 minute. It is ready only when the last line says **Ready! / 準備好了！** Seeing only warnings in the middle is normal, and you may not see progress bars.
這格只要跑一次，大約 1 分鐘。最後一行出現 **Ready! / 準備好了！** 才算好；中間只看到警告是正常的，不一定有進度條。

In [ ]:
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # no account key needed / 不用金鑰
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print("transformers", transformers.__version__, "| torch", torch.__version__)
try:
    tok4 = AutoTokenizer.from_pretrained("google/gemma-4-E2B-it")          # Gemma 4 tokenizer only
    tok = AutoTokenizer.from_pretrained("unsloth/gemma-3-270m-it")         # small model's tokenizer
    model = AutoModelForCausalLM.from_pretrained("unsloth/gemma-3-270m-it",
              dtype=torch.float32, attn_implementation="eager").eval()
    print("Ready! / 準備好了！")
except Exception as e:
    print("Something went wrong:", repr(e)[:300])
    print("Next step: menu Runtime -> Restart session, then run this cell again. "
          "If it still fails, raise your hand. / 下一步：執行階段 → 重新啟動工作階段，再跑一次；還是不行就舉手。")

## Step 1 Tokenization / 斷詞

**Question: How does text become numbers? / 這一步回答：文字怎麼變成數字？**

Tokenizer here: **real Gemma 4 tokenizer** (`google/gemma-4-E2B-it`). No model is used in this step.
這一步用的斷詞器：**真的 Gemma 4 斷詞器**。這一步不用模型。

**Guess first / 先猜：** How many tokens is *"Hualien is famous for"*? 4 words → 4 tokens? More? Fewer?
*"Hualien is famous for"* 會切成幾個 token？4 個字就 4 個？更多？更少？

In [ ]:
def show_tokens(text):
    ids = tok4(text, add_special_tokens=False)["input_ids"]
    pieces = tok4.convert_ids_to_tokens(ids)
    print(f"{text!r} -> {len(ids)} tokens")
    for i, p in zip(ids, pieces):
        print(f"  ID {i:>7}  {p!r}")
    return ids

show_tokens("Hualien is famous for")

The `▁` mark means "a space comes before this piece". When `▁` is a piece on its own, it is just one space.
「▁」代表這一塊前面有一個空格；`▁` 單獨一塊時，它本身就是一個空格。

**Check against the official guide / 跟官方導覽對答案**: the code below *tests* the numbers in the official guide. If every line says PASS, all four items match what the official guide says.
下面這格會**檢查**官方導覽寫的數字。每一行都是 PASS，代表這四項都跟官方導覽寫的一樣。

In [ ]:
checks = {
    '"The cat sat on the mat." IDs': tok4("The cat sat on the mat.", add_special_tokens=False)["input_ids"] == [818, 5866, 2838, 580, 506, 1756, 236761],
    'vocabulary size = 262,144':      len(tok4) == 262144,
    '<|turn> = 105':                  tok4.convert_tokens_to_ids("<|turn>") == 105,
    '<turn|> = 106':                  tok4.convert_tokens_to_ids("<turn|>") == 106,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)
assert all(checks.values()), "Some items do not match the official guide / 有項目跟官方導覽寫的不一樣"
print("Vocabulary size / 詞表大小:", f"{len(tok4):,}")

**Try other kinds of text / 試試別種文字**: Chinese, emoji, a number, and a rare character.
中文、emoji、數字、罕用字。

Common characters have their own number. A rare character that is not in the vocabulary (like 鬮) is broken into the few bytes the computer uses to store it, one byte per piece, shown as `<0x..>` (byte fallback).
常用字有自己的編號；詞表裡沒有的罕用字（例如「鬮」）會被拆成電腦儲存它用的幾個位元組，一個位元組一塊，顯示成 `<0x..>`（byte fallback）。

Also notice how "2026" is split, with and without a word before it (is there a `▁` piece?).
也看看「2026」被怎麼切：前面有字、和單獨一個「2026」，有沒有 `▁` 那一塊？

(Details, if you are curious / 想知道細節再看: the official guide says a byte piece's ID = 238 + byte value, e.g. `<0xE9>` = 238 + 233 = 471. 官方導覽寫：位元組那一塊的 ID ＝ 238 ＋ 位元組的值，例如 `<0xE9>` ＝ 238 ＋ 233 ＝ 471。)

In [ ]:
for s in ["花蓮的海很藍", "加鬮", "I love 🍉 in 2026", "2026"]:
    show_tokens(s)
    print()

**Count: characters vs tokens / 數一數：字數和 token 數**

"space-separated chunks" = how many pieces you get by cutting at spaces. 「space-separated chunks」＝用空白切出的段數。

In [ ]:
for s in ["Hualien is famous for", "花蓮的海很藍", "I love 🍉 in 2026"]:
    n_tok = len(tok4(s, add_special_tokens=False)["input_ids"])
    print(f"{s!r}: {len(s)} characters, {len(s.split())} space-separated chunks, {n_tok} tokens")

**Special tokens / 特殊記號**: some IDs are not words, they are control marks for the chat format (we use them in Step 5).
有些 ID 不是字，而是聊天格式用的控制記號（第 5 步會用到）。

`<bos>` = the "start" mark at the very front of a text; it is not a word. 「`<bos>`」＝句子最前面的「開始」記號，不是一個字。

In [ ]:
for t in ["<bos>", "<eos>", "<|turn>", "<turn|>", "<|think|>", "<unk>"]:
    print(f"{t:10} -> ID {tok4.convert_tokens_to_ids(t)}")

**Your turn / 換你試** ✏️ Change the sentence in the line `my_sentence = ...` to your own sentence (any language), then run it again.
把 `my_sentence = ...` 那一行改成你自己的句子（任何語言都可以），再跑一次。
After you change it, rerun only this cell (the ▶ on its left); no need for Run all. Keep the quotes and commas when you edit.
改完只要重跑這一格（按這格左邊的 ▶），不用再按全部執行；改的時候引號和逗號要留著。

In [ ]:
my_sentence = "I am a student at NDHU in Hualien."   # <- change this line / 改這一行
show_tokens(my_sentence);

## Step 2 Embeddings / 嵌入向量

**Question: How do numbers carry meaning? / 這一步回答：數字怎麼帶有意思？**

Model here: **Gemma 3 270M** (smaller model of the same family; ideas are the same, numbers differ from the official Gemma 4 guide). Each token ID is looked up in a table and becomes a list of **640** numbers (a vector). We use that first lookup table (the input embedding), before any layer runs.
這一步用的模型：**Gemma 3 270M**（比較小的同家族模型；道理一樣、數字跟官方導覽不一樣）。每個 token ID 查表後變成一串 **640** 個數字（向量）。這裡用的是最開頭那張查表（輸入嵌入），還沒經過任何一層。

**Guess first / 先猜：** Which word is closest to *cat*: kitten, dog, car, truck, or banana?
*cat* 跟哪個字最接近：kitten、dog、car、truck、banana？

In [ ]:
emb = model.get_input_embeddings().weight.detach()
def word_vec(word):
    ids = tok(" " + word, add_special_tokens=False)["input_ids"]   # a word may be several pieces / 一個字可能切成好幾塊
    return emb[ids].mean(0)                                       # average them / 把幾塊平均

v = word_vec("cat")
print("cat -> vector of", len(v), "numbers. First 8:")
print([round(x, 3) for x in v[:8].tolist()])

These numbers mean nothing on their own. They only mean something when we compare two words (next cell).
這些數字單獨看沒有意義，要兩個字一起比才有意義（下一格）。

**Cosine similarity / 餘弦相似度**: −1 = opposite directions, 0 = perpendicular, +1 = same direction.
−1＝方向相反、0＝互相垂直、+1＝方向相同。

If a word is cut into several pieces, we first average the pieces' vectors, then compare. So these numbers do not measure the same thing as the 0.43 in the official guide (Gemma 4 E2B's starting embedding table).
一個字被切成好幾塊時，這裡先把幾塊的向量平均再比；所以這裡的數字跟官方導覽 E2B 起始嵌入表的 0.43 量的不是同一件事。

In [ ]:
import pandas as pd
import torch.nn.functional as F
words = ["cat", "kitten", "dog", "car", "truck", "banana"]
words.append("tiger")             # <- change this word / 改成你想加的字
vecs = torch.stack([word_vec(w) for w in words])
sim = F.cosine_similarity(vecs[:, None, :], vecs[None, :, :], dim=-1)
pd.DataFrame(sim.numpy().round(2), index=words, columns=words)

The result may surprise you (for example, *cat* may be closer to *dog* than to *kitten*). This is a small model's first lookup table, so describe what you see honestly. The official guide's numbers (e.g. cat–kitten 0.43) come from Gemma 4 and will not match.
結果可能出乎意料（例如 cat 跟 dog 比跟 kitten 還像）。這是小模型的第一張查表，照實描述就好。官方導覽的數字（例如 cat–kitten 0.43）是 Gemma 4 的，不會跟這裡一樣。

How to read the table: find the `tiger` row, skip the 1.00 (a word compared with itself); the column with the biggest number is the most similar word.
怎麼看這張表：找到 tiger 那一列，跳過 1.00（自己跟自己），數字最大的那一欄就是最像的字。

**Your turn / 換你試** ✏️ Change `"tiger"` to another English word (e.g. `"bus"`, `"apple"`) and run again. Which word is it closest to?
把 `"tiger"` 換成別的英文字（例如 `"bus"`、`"apple"`）再跑，它跟誰最接近？
After you change it, rerun only this cell (the ▶ on its left); no need for Run all. Keep the quotes and commas when you edit.
改完只要重跑這一格（按這格左邊的 ▶），不用再按全部執行；改的時候引號和逗號要留著。

## Step 3 Attention / 注意力

**Question: How does a word look at other words? / 這一步回答：一個字怎麼「看」其他字？**

Model here: **Gemma 3 270M** (18 layers, 4 attention heads per layer). We feed one sentence, then read the real attention numbers of the word **"it"**: how much "it" looks at itself and at each earlier piece. We show **the last layer (layer 18), averaged over its 4 heads**.
這一步用的模型：**Gemma 3 270M**（18 層，每層 4 個注意力頭）。讀出 **it** 這個字看它自己和前面的每一塊的真實注意力數字。畫的是**最後一層（第 18 層）、4 個頭取平均**。

**Guess first / 先猜：** In *"The trophy didn't fit in the suitcase because it was too big."*, does "it" look most at *trophy* or *suitcase*?
「it」最注意的是 trophy 還是 suitcase？

A common result: neither gets much; most attention goes to the start mark at the front of the sentence. That itself is a finding worth writing down.
常見的結果是兩個都不多，大部分跑到句首記號，這本身就是值得寫下來的發現。

In [ ]:
import matplotlib.pyplot as plt
sentence = "The trophy didn't fit in the suitcase because it was too big."
LAYER = 18                                    # <- try 1, 6, 12, 18 / 改這一行試別層
enc = tok(sentence, return_tensors="pt")
with torch.no_grad():
    att = model(**enc, output_attentions=True).attentions[LAYER - 1][0].mean(0)  # avg heads
pieces = [tok.decode([i]) for i in enc["input_ids"][0]]
pos = [p.strip() for p in pieces].index("it")
row = att[pos, :pos + 1].numpy()
plt.figure(figsize=(10, 3)); plt.bar(range(len(row)), row)
plt.xticks(range(len(row)), [repr(p) for p in pieces[:pos + 1]], rotation=60)
plt.title(f'Attention from "it" (layer {LAYER}, mean of 4 heads)'); plt.ylabel("weight"); plt.show()
print("Top 5 / 前 5 名:")
for p, w in sorted(zip(pieces, row), key=lambda x: -x[1])[:5]:
    print(f"{p!r:>12}  {w:.3f}")
print("trophy and suitcase / 這兩塊:")
for w in ["trophy", "suitcase"]:
    hits = [k for k, p in enumerate(pieces[:pos + 1]) if p.strip() == w]
    print(f"{w!r:>12}  " + (f"{row[hits[0]]:.3f}" if hits else "(split into several pieces / 被切成好幾塊)"))

**What do you see? / 你看到什麼？** Describe it honestly. The real numbers may not match the textbook story ("it" → trophy). Often a lot of attention goes to the very first token `<bos>` (it works like a "parking spot"). Real models are messier than diagrams.
照實描述就好。真實數字不一定像課本的故事（it → trophy）。常常很多注意力落在第一個記號 `<bos>`（像個「停車位」）。真的模型比示意圖亂。

**Your turn / 換你試** ✏️ Change `LAYER = 18` to `1`, `6`, or `12` and run again. Does the picture change?
把 `LAYER = 18` 改成 `1`、`6` 或 `12` 再跑一次，圖有變嗎？
After you change it, rerun only this cell (the ▶ on its left); no need for Run all. Keep the quotes and commas when you edit.
改完只要重跑這一格（按這格左邊的 ▶），不用再按全部執行；改的時候引號和逗號要留著。

## Step 4 Sampling / 取樣

**Question: How is the next word chosen? / 這一步回答：下一個字怎麼選出來？**

Model here: **Gemma 3 270M**. The model gives one score (logit) for every token in its vocabulary; softmax turns scores into probabilities. These are real numbers from this small model, not from Gemma 4.
這一步用的模型：**Gemma 3 270M**。模型對詞表裡每個 token 給一個分數（logit），softmax 把分數變成機率。這些是這個小模型的真實數字，不是 Gemma 4 的。

**Guess first / 先猜：** What is the most likely next token after *"Hualien is famous for"*?
*"Hualien is famous for"* 下一個最可能是什麼？

In [ ]:
prompt = "Hualien is famous for"
with torch.no_grad():
    logits = model(**tok(prompt, return_tensors="pt")).logits[0, -1]
probs = torch.softmax(logits, -1)
top = probs.topk(10)
print("vocabulary scores:", len(logits))
for rank, (p, i) in enumerate(zip(top.values, top.indices), 1):
    print(f"{rank:>2}. {tok.decode([int(i)])!r:14} {float(p):7.2%}")

**Temperature, top-k, top-p** (official Gemma 4 defaults: 1.0 / 64 / 0.95). 官方預設值：temperature 1.0、top-k 64、top-p 0.95。

Order / 順序: scores ÷ temperature → softmax → top-k → top-p → pick one at random.
分數除以溫度 → softmax → 只留前 k 名 → 再留到累加第一次達到或超過 p → 隨機選一個。

- temperature: divide scores before softmax. Low = sharper, high = flatter. 0 = always pick the top one (greedy).
  低＝更集中、高＝更平均；0＝每次都選第一名。
- top-k: keep only the k best tokens, then rescale them to 100%. 只留前 k 名，再重新算成 100%。
- top-p: add probabilities from the highest down; keep tokens until the total first reaches or exceeds p, then rescale to 100%.
  由高到低累加機率，留到第一次達到或超過 p，再重新算成 100%。

The next cell only builds two tools (`sample_next`, `continue_text`). It prints one OK line; that is normal.
下一格只是先把兩個工具做好，跑完只印一行 OK，這是正常的。

In [ ]:
def sample_next(logits, temperature=1.0, top_k=64, top_p=0.95):
    if temperature == 0:
        return int(logits.argmax())                  # 0 = always the top one (greedy)
    probs = torch.softmax(logits / temperature, -1)
    p, idx = probs.topk(top_k)                       # top-k
    p = p / p.sum()                                  # rescale to 100%
    keep = (p.cumsum(0) - p) < top_p                 # until total first reaches >= p
    p, idx = p[keep], idx[keep]
    p = p / p.sum()                                  # rescale to 100% again
    return int(idx[torch.multinomial(p, 1)])

def continue_text(prompt, n_tokens=6, **kw):
    ids = tok(prompt, return_tensors="pt")["input_ids"]
    for _ in range(n_tokens):
        with torch.no_grad():
            nxt = sample_next(model(input_ids=ids).logits[0, -1], **kw)
        ids = torch.cat([ids, torch.tensor([[nxt]])], 1)
    return tok.decode(ids[0], skip_special_tokens=True)

print("OK: sample_next and continue_text are ready / 兩個工具做好了")

In [ ]:
for T in [0.2, 1.5]:                       # <- try other temperatures / 改溫度試試
    print(f"--- temperature = {T} ---")
    for k in range(10):
        print(continue_text("Hualien is famous for", temperature=T))

**Your turn / 換你試** ✏️ Change `[0.2, 1.5]` to `[0, 0.7, 3.0]` and run again. Or paste this line into a new cell and run it a few times: `print(continue_text("Hualien is famous for", top_k=1))`. What happens with top_k=1?
把 `[0.2, 1.5]` 改成 `[0, 0.7, 3.0]` 再跑一次；或把上面那一行貼到新的一格（按 **+ Code** 新增一格）、多跑幾次。top_k=1 會怎樣？
After you change it, rerun only this cell (the ▶ on its left); no need for Run all. Keep the quotes and commas when you edit.
改完只要重跑這一格（按這格左邊的 ▶），不用再按全部執行；改的時候引號和逗號要留著。

## Step 5 Chat / 聊天

**Question: How does a chat become one long text? / 這一步回答：聊天怎麼變成一長串文字？**

The cell below prints both templates. First, the **real Gemma 4 tokenizer's chat template**: see the control marks `<|turn>` and `<turn|>`. Then the template of the **small Gemma 3 270M**, which uses the older Gemma 3 marks `<start_of_turn>` / `<end_of_turn>`. Same idea, different mark names. The next cell uses the small Gemma 3 270M to generate the answer.
這一格兩個範本都印出來：先是**真的 Gemma 4 斷詞器的聊天範本**，看控制記號 `<|turn>`、`<turn|>`；再是**小模型 Gemma 3 270M** 的範本，它用舊版 Gemma 3 的記號 `<start_of_turn>`／`<end_of_turn>`。道理一樣，記號名字不同。下一格用小模型 Gemma 3 270M 生成。

**Guess first / 先猜：** When you chat with an AI, does the model "see" separate messages, or one long text?
你跟 AI 聊天時，模型看到的是一則一則訊息，還是一整串文字？

In [ ]:
messages = [{"role": "user", "content": "What is Hualien famous for?"}]
raw4 = tok4.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== Gemma 4 raw prompt ===");  print(raw4)
raw3 = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== Gemma 3 (small model) raw prompt ===");  print(raw3)

**Watch the loop / 看自迴歸迴圈**: the model makes one token, adds it to the text, and goes again, until it writes the end mark or reaches the limit of 80 pieces.
模型生一個 token、接到文字後面、再生下一個，直到寫出結束記號，或到 80 塊的上限。

In [ ]:
ids = tok(raw3, return_tensors="pt", add_special_tokens=False)["input_ids"]
stop = {tok.convert_tokens_to_ids("<end_of_turn>"), tok.eos_token_id}
past, x = None, ids
for step in range(80):                                    # at most 80 tokens
    with torch.no_grad():
        out = model(input_ids=x, past_key_values=past, use_cache=True)
    past = out.past_key_values
    nxt = sample_next(out.logits[0, -1], temperature=1.0, top_k=64, top_p=0.95)
    if nxt in stop:
        print(f"\n[stop mark {tok.decode([nxt])!r} after {step} tokens]"); break
    print(tok.decode([nxt]), end="|", flush=True)         # "|" shows token borders
    x = torch.tensor([[nxt]])
else:
    print("\n[reached the 80-piece limit / 到 80 塊的上限]")

The `|` marks show where each token ends. A 270M model is tiny, so its answer may be wrong or strange. That is a good reminder: fluent text is not the same as true facts.
`|` 標出每個 token 的邊界。270M 是很小的模型，回答可能錯或怪——這正好提醒我們：講得順不等於講得對。

**Your turn / 換你試** ✏️ Scroll up to the cell that prints `=== Gemma 4 raw prompt ===`. Change the question inside `messages = ...` there. Then rerun that cell and the cell right below it (each ▶ on its left); no need for Run all. Keep the quotes and commas when you edit.
往上找會印出 `=== Gemma 4 raw prompt ===` 的那一格，改裡面 `messages = ...` 的句子；那一格和它下面那一格都要重跑（各按左邊的 ▶），不用再按全部執行；改的時候引號和逗號要留著。

## Look back / 回顧

Think about these on your own, using what you saw today. 用今天看到的東西，自己想一想：

1. Step 1: how many pieces was your own sentence cut into? 你的句子被切成幾塊？
2. Step 2: which word was closest to cat? 誰跟 cat 最近？
3. Step 3: which piece did "it" look at most? it 最看哪一塊？
4. Step 4: how were the results at the two temperatures different? 兩個溫度差在哪？
5. Step 5: which marks did you see in the raw chat string? 原始聊天字串裡有哪些記號？

Source of the official numbers / 官方數字來源: Google, *How LLMs work (with Gemma 4)*, https://gemma.ai/learn/how-llms-work/